# Phase 6: Baseline vs Self-Healing RAG Evaluation

This notebook evaluates the fixed Phase 6 labels without changing Phase 5 thresholds. Every model runs locally in Google Colab; no paid API or LLM-as-a-judge service is used.

## Environment

Replace `REPOSITORY_URL` with your repository URL before running this notebook in Colab.

In [ ]:
from pathlib import Path
import os
import random
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/YOUR_GITHUB_USERNAME/adaptive-rag.git'
PROJECT_DIR = Path('/content/adaptive-rag')

if not (PROJECT_DIR / 'src').exists():
    if 'YOUR_GITHUB_USERNAME' in REPOSITORY_URL:
        raise RuntimeError('Set REPOSITORY_URL to your GitHub repository URL, then run this cell again.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import numpy as np
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')

## Initialize Phase 1–5 components

The baseline uses dense top-3 retrieval and always follows its normal generation path. The self-healing system uses the existing detector and LangGraph workflow. Both share one local Qwen instance.

In [ ]:
from src.rag import (
    BM25Retriever,
    BaselineRAG,
    CrossEncoderReranker,
    FAISSRetriever,
    HybridRetriever,
    LocalQwenGenerator,
    RAGConfig,
    RetrievalFailureDetector,
    SelfHealingRAGWorkflow,
    SelfHealingWorkflowConfig,
    load_documents,
)

rag_config = RAGConfig(
    embedding_model_name='sentence-transformers/all-MiniLM-L6-v2',
    generation_model_name='Qwen/Qwen2.5-1.5B-Instruct',
    top_k=3,
    max_new_tokens=120,
)
documents = load_documents(Path('data/phase1_corpus.json'))
index_dir = Path('data/phase1_faiss_index')

dense_retriever = FAISSRetriever(
    embedding_model_name=rag_config.embedding_model_name,
    device=rag_config.device,
    batch_size=rag_config.embedding_batch_size,
)
if (index_dir / 'documents.faiss').exists():
    dense_retriever.load(index_dir)
else:
    dense_retriever.build(documents)
    dense_retriever.save(index_dir)

bm25_retriever = BM25Retriever(documents)
hybrid_retriever = HybridRetriever(dense_retriever, bm25_retriever)
reranker = CrossEncoderReranker(device=rag_config.device)
generator = LocalQwenGenerator(rag_config)
baseline_rag = BaselineRAG(dense_retriever, generator, rag_config)
self_healing_workflow = SelfHealingRAGWorkflow(
    dense_retriever=dense_retriever,
    bm25_retriever=bm25_retriever,
    hybrid_retriever=hybrid_retriever,
    reranker=reranker,
    generator=generator,
    failure_detector=RetrievalFailureDetector(),
    config=SelfHealingWorkflowConfig(max_retries=1),
)
print('Components initialized. Qwen device:', generator.device)

## Load the fixed labeled set

The dataset contains five examples in each category. Missing-evidence queries have no relevant document IDs and are expected to abstain.

In [ ]:
from collections import Counter
from src.evaluation import EvaluationRunner, load_evaluation_set

examples = load_evaluation_set(Path('data/phase6_evaluation.json'))
print('Examples:', len(examples))
print('Categories:', dict(Counter(example.category for example in examples)))

## Run evaluation

This runs local generation twice per example in the ordinary case, so the full set may take several minutes on a free Colab GPU.

In [ ]:
runner = EvaluationRunner(
    baseline_rag=baseline_rag,
    self_healing_workflow=self_healing_workflow,
    baseline_top_k=3,
)

def show_progress(index, total, example):
    print(f'[{index:02d}/{total:02d}] {example.id}: {example.query}')

report = runner.run(examples, progress_callback=show_progress)
print('Evaluation complete.')

## Overall metrics and baseline comparison

Baseline failure classification treats every query as `HEALTHY`, because the baseline has no detector. Groundedness is a lexical-support heuristic with a fixed 0.50 coverage threshold, not a semantic judge.

In [ ]:
metric_names = [
    'failure_classification_accuracy',
    'abstention_accuracy',
    'retrieval_hit_rate',
    'recovery_success_rate',
    'unnecessary_recovery_rate',
    'average_retry_count',
    'answered_percentage',
    'abstained_percentage',
    'average_groundedness_score',
    'grounded_answer_rate',
    'expected_keyword_hit_rate',
]

baseline_metrics = report.baseline_metrics.to_dict()
self_healing_metrics = report.self_healing_metrics.to_dict()
print(f'{"Metric":38} {"Baseline":>12} {"Self-healing":>14}')
print('-' * 68)
for name in metric_names:
    baseline_value = baseline_metrics[name]
    self_healing_value = self_healing_metrics[name]
    baseline_text = 'N/A' if baseline_value is None else f'{baseline_value:.4f}'
    self_healing_text = 'N/A' if self_healing_value is None else f'{self_healing_value:.4f}'
    print(f'{name:38} {baseline_text:>12} {self_healing_text:>14}')

## Per-category results

In [ ]:
for category in sorted(report.self_healing_metrics.per_category):
    print('\n' + category.upper())
    baseline_category = report.baseline_metrics.per_category[category]
    self_healing_category = report.self_healing_metrics.per_category[category]
    for name in metric_names:
        baseline_value = baseline_category[name]
        self_healing_value = self_healing_category[name]
        baseline_text = 'N/A' if baseline_value is None else f'{baseline_value:.4f}'
        self_healing_text = 'N/A' if self_healing_value is None else f'{self_healing_value:.4f}'
        print(f'  {name:36} baseline={baseline_text:>8}  self-healing={self_healing_text:>8}')

## Failed examples for inspection

In [ ]:
for system in ('baseline', 'self_healing'):
    failed = report.failed_examples(system)
    print(f'\n{system.upper()} FAILED EXAMPLES: {len(failed)}')
    for record in failed:
        print(f'\n{record.example_id} [{record.category}] {record.query}')
        print(f'  expected/predicted: {record.expected_failure_type} / {record.predicted_failure_type}')
        print(f'  retrieved: {list(record.retrieved_document_ids)}')
        print(f'  path: {" → ".join(record.path)}')
        print(f'  reasons: {list(record.failure_reasons)}')
        print(f'  answer: {record.answer[:300]}')